## AI Text Analysis Tool using Pre-trained Transformer Models

# Objectives

The main objective of this project is to understand and apply modern Natural Language Processing (NLP) techniques using pre-trained transformer models. The project aims to perform multiple NLP tasks such as sentiment analysis, named entity recognition (NER), and text summarization using models from Hugging Face. Another key objective is to compare these advanced models with traditional NLP methods implemented in Day 9, which involved preprocessing and machine learning techniques. This project also focuses on demonstrating the concept of transfer learning, where models trained on large datasets can be directly used for predictions without additional training.

# Literature Review

Traditional NLP techniques rely heavily on preprocessing steps such as tokenization, stopword removal, stemming, and lemmatization. These methods convert text into numerical features using techniques like TF-IDF, followed by machine learning models such as Logistic Regression or Naive Bayes. While effective, these approaches require manual feature engineering and often fail to capture contextual meaning.

Modern NLP is based on transformer architectures such as BERT, GPT, and T5. These models are pre-trained on massive datasets and can understand context, semantics, and relationships between words. This approach is known as transfer learning, where a model trained on one task is reused for another. Transformer models eliminate the need for heavy preprocessing and provide more accurate and efficient results.

In [30]:
import os
os.listdir()

['AI-text-Analysis-tool.ipynb', 'Hugging-Face.ipynb', 'Reviews.csv']

## Step 1: Data Loading

In this step, the dataset is loaded using the Pandas library, which is widely used for data manipulation and analysis. The 'Text' column is selected as it contains the customer reviews required for NLP tasks. A small random sample of 10 reviews is taken to ensure faster execution and efficient performance on a low-resource system.

Technique used: Data Sampling  
Data sampling is the process of selecting a small subset of data from a large dataset for faster computation.

Result: A list of review texts (`text_data`) is created, which will be used as input for further NLP tasks.

**Conclusion:** This step prepares the raw input data for analysis without any preprocessing, showcasing the advantage of modern NLP models.

In [31]:
# Step 1- Data Loading
import pandas as pd

data = pd.read_csv("Reviews.csv")

# Check columns
print(data.columns)

# Take small RANDOM sample (better)
text_data = data["Text"].dropna().sample(10, random_state=42).tolist()

for i, text in enumerate(text_data):
    print(f"{i+1}. {text[:100]}...")

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='str')
1. Having tried a couple of other brands of gluten-free sandwich cookies, these are the best of the bun...
2. My cat loves these treats. If ever I can't find her in the house, I just pop the top and she bolts o...
3. A little less than I expected.  It tends to have a muddy taste - not what I expected since they said...
4. First there was Frosted Mini-Wheats, in original size, then there was Frosted Mini-Wheats Bite Size....
5. and I want to congratulate the graphic artist for putting the entire product name on such a small bo...
6. Please add more Pineapple flavor to your packages of lifesavers, in fact, can you sell pineapple fla...
7. I absolutely love Yorkshire tea and am so glad it is now available on Amazon!  A cup of this full-bo...
8. I have such a hard time finding loose tea locally. Being able to order my favorite L

## Step 2: Sentiment Analysis

In this step, sentiment analysis is performed using a pre-trained transformer model from Hugging Face. The model is based on BERT (Bidirectional Encoder Representations from Transformers), which understands the context of words by considering both left and right surroundings in a sentence.

Technique Used: Transfer Learning
Transfer learning is a method where a model trained on a large dataset is reused for a different task without training from scratch.

Tools Used: Hugging Face Transformers (pipeline)

Result: The model predicts whether each review is positive or negative along with a confidence score. The predictions are generally highly confident, indicating strong performance of transformer models.

In [32]:
# Step 2- Sentiment Analysis (Pre-trained Model)

from transformers import pipeline


sentiment_model = pipeline("sentiment-analysis", device=-1)

print("\n--- Sentiment Analysis Results ---\n")

for i, text in enumerate(text_data):
    result = sentiment_model(text[:512])[0]   # limit length
    
    print(f"Review {i+1}:")
    print(f"Sentiment: {result['label']}")
    print(f"Confidence: {round(result['score'], 4)}")
    print("=" * 50)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2264.93it/s]



--- Sentiment Analysis Results ---

Review 1:
Sentiment: POSITIVE
Confidence: 0.9992
Review 2:
Sentiment: POSITIVE
Confidence: 0.9975
Review 3:
Sentiment: NEGATIVE
Confidence: 0.9994
Review 4:
Sentiment: POSITIVE
Confidence: 0.9377
Review 5:
Sentiment: POSITIVE
Confidence: 0.9976
Review 6:
Sentiment: POSITIVE
Confidence: 0.9956
Review 7:
Sentiment: POSITIVE
Confidence: 0.9998
Review 8:
Sentiment: POSITIVE
Confidence: 0.985
Review 9:
Sentiment: NEGATIVE
Confidence: 0.9936
Review 10:
Sentiment: POSITIVE
Confidence: 0.976


## Step 3: Named Entity Recognition (NER)

Named Entity Recognition (NER) is used to identify and classify entities such as organizations, locations, and person names within the text. A pre-trained transformer model is used to perform token classification.

Technique Used: Token Classification
Token classification assigns labels to each word (token) in a sentence, such as ORG (organization), PER (person), or LOC (location).

Since transformer models use subword tokenization, post-processing is applied to clean the output by removing incomplete tokens and low-confidence predictions.

Tools Used: Hugging Face Transformers (pipeline)

Result: Meaningful entities such as company names (e.g., Amazon, Whole Foods) are extracted, providing structured information from unstructured text.

In [33]:
# Step 3 - Named Entity Recognition (NER)
from transformers import pipeline

# Load NER model (with aggregation to merge tokens)
ner = pipeline("ner", aggregation_strategy="simple", device=-1)

# ------------------------------
# Function to clean NER output
# ------------------------------
def clean_ner_output(entities):
    cleaned = []
    
    for e in entities:
        word = e['word']
        
        # Remove broken subwords (## tokens)
        if word.startswith("##"):
            continue
        
        # Remove very small meaningless tokens
        if len(word.strip()) <= 2:
            continue
        
        # Remove low confidence predictions
        if float(e['score']) < 0.6:
            continue
        
        # ✅ Keep useful entities
        cleaned.append({
            "entity": word,
            "type": e['entity_group'],
            "confidence": round(float(e['score']), 3)
        })
    
    return cleaned


# ------------------------------
# Apply NER on sample text data
# ------------------------------
print("\n--- Cleaned Named Entity Recognition (NER) ---\n")

for i, text in enumerate(text_data):
    
    # Limit text length (important for speed)
    raw_entities = ner(text[:512])
    
    # Clean output
    clean_entities = clean_ner_output(raw_entities)
    
    print(f"Review {i+1}:")
    
    if clean_entities:
        for entity in clean_entities:
            print(f"Entity: {entity['entity']} | Type: {entity['type']} | Confidence: {entity['confidence']}")
    else:
        print("No meaningful entities found")
    
    print("=" * 60)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1855.53it/s]
BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Cleaned Named Entity Recognition (NER) ---

Review 1:
Entity: Glutino | Type: ORG | Confidence: 0.731
Review 2:
No meaningful entities found
Review 3:
No meaningful entities found
Review 4:
Entity: Mini | Type: MISC | Confidence: 0.621
Entity: Cinnamon Roll | Type: MISC | Confidence: 0.956
Review 5:
Entity: Gold | Type: PER | Confidence: 0.779
Review 6:
No meaningful entities found
Review 7:
Entity: Yorkshire | Type: MISC | Confidence: 0.608
Entity: Amazon | Type: ORG | Confidence: 0.946
Review 8:
Entity: Lady Grey | Type: MISC | Confidence: 0.931
Entity: Amazon | Type: LOC | Confidence: 0.992
Review 9:
No meaningful entities found
Review 10:
Entity: Trader Joe ' s | Type: ORG | Confidence: 0.984
Entity: Whole Foods | Type: ORG | Confidence: 0.997


## Step 4- Abstractive text Summarization

In this step, text summarization is performed using the T5 (Text-to-Text Transfer Transformer) model. T5 converts all NLP tasks into a text-to-text format, where both input and output are text.

Technique Used: Abstractive Summarization
Abstractive summarization generates new sentences that capture the main idea of the text rather than simply extracting parts of it.

Due to compatibility and stability considerations, the model is used directly with AutoTokenizer and AutoModelForSeq2SeqLM. Beam search is applied to improve the quality of generated summaries.

Tools Used: Hugging Face Transformers (T5 model)

Result: The model generates concise summaries that capture the key meaning of each review.

In [34]:
# Step 4 - (Text Summarization) T5 model

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

print("\n--- Abstractive Summarization ---\n")

for i, text in enumerate(text_data):
    
    input_text = "summarize: " + text[:512]
    
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=60,
        min_length=15,
        num_beams=4,
        early_stopping=True
    )
    
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"Review {i+1}:")
    print(f"Original: {text[:120]}...")
    print(f"Summary: {summary}")
    print("=" * 60)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2027.98it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



--- Abstractive Summarization ---

Review 1:
Original: Having tried a couple of other brands of gluten-free sandwich cookies, these are the best of the bunch.  They're crunchy...
Summary: I've tried a few other gluten-free cookies, but I'm not sure if it's a gluten-free cookie or a gluten-free cookie.
Review 2:
Original: My cat loves these treats. If ever I can't find her in the house, I just pop the top and she bolts out of wherever she w...
Summary: Makes a great gift for a cat that doesn't just love treats.
Review 3:
Original: A little less than I expected.  It tends to have a muddy taste - not what I expected since they said it was the favorite...
Summary: It's a little less than I expected. It tends to have a muddy taste - not what I expected since they said it was the favorite.
Review 4:
Original: First there was Frosted Mini-Wheats, in original size, then there was Frosted Mini-Wheats Bite Size. Well, if for some r...
Summary: Bite Size is a little bit smaller than the Bite Siz

## Step 5: Extractive Summarization

In addition to abstractive summarization, a simple extractive summarization method is implemented as a fallback.

Technique Used: Extractive Summarization
Extractive summarization selects important sentences directly from the original text instead of generating new content.

Tools Used: Python string operations

Result: The first meaningful sentence of each review is extracted as a quick and efficient summary, especially useful for low-resource environments.

In [35]:
# Step 5- Extractive Summarization

print("\n--- Extractive Summarization ---\n")

for i, text in enumerate(text_data):
    
    summary = text.split(".")[0]
    
    print(f"Review {i+1}:")
    print(f"Original: {text[:120]}...")   # ✅ show review
    print(f"Summary: {summary}")
    print("=" * 60)


--- Extractive Summarization ---

Review 1:
Original: Having tried a couple of other brands of gluten-free sandwich cookies, these are the best of the bunch.  They're crunchy...
Summary: Having tried a couple of other brands of gluten-free sandwich cookies, these are the best of the bunch
Review 2:
Original: My cat loves these treats. If ever I can't find her in the house, I just pop the top and she bolts out of wherever she w...
Summary: My cat loves these treats
Review 3:
Original: A little less than I expected.  It tends to have a muddy taste - not what I expected since they said it was the favorite...
Summary: A little less than I expected
Review 4:
Original: First there was Frosted Mini-Wheats, in original size, then there was Frosted Mini-Wheats Bite Size. Well, if for some r...
Summary: First there was Frosted Mini-Wheats, in original size, then there was Frosted Mini-Wheats Bite Size
Review 5:
Original: and I want to congratulate the graphic artist for putting the entire produ

Step 6: Final AI Text Analysis Tool

In this step, all the components—sentiment analysis, NER, and summarization—are combined into a single function to create a unified AI tool.

Technique Used: Pipeline Integration
Multiple NLP tasks are integrated into a single workflow to provide comprehensive text analysis.

Tools Used: Python functions, Hugging Face models

Result: The tool takes any input text and returns:

Sentiment (Positive/Negative)
Confidence score
Extracted entities
Generated summary

This demonstrates a real-world application of NLP.

In [36]:
# Step 6- Final AI Text Analysis Tool

def analyze_text(text):
    
    text = text[:512]
    
    # Sentiment
    sentiment = sentiment_model(text)[0]
    
    # NER
    entities = clean_ner_output(ner(text))
    
    # Summarization
    input_text = "summarize: " + text
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=60,
        min_length=15,
        num_beams=4,
        early_stopping=True
    )
    
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return {
        "Sentiment": sentiment['label'],
        "Confidence": round(sentiment['score'], 3),
        "Entities": entities,
        "Summary": summary
    }

print(result)

{'label': 'POSITIVE', 'score': 0.9760100245475769}


In [37]:
# Step 7- Test

test_text = "I bought this product from Amazon and it is amazing. I use it daily."

result = analyze_text(test_text)

print("\n--- Final Output ---\n")

for key, value in result.items():
    print(f"{key}: {value}")


--- Final Output ---

Sentiment: POSITIVE
Confidence: 1.0
Entities: [{'entity': 'Amazon', 'type': 'ORG', 'confidence': 0.99}]
Summary: I love this product. It's a great product. It's easy to use and easy to use.


## Step 7- Test

The final AI text analysis tool successfully processes input text and provides meaningful insights using multiple NLP techniques. The tool was tested on a sample review, and it generated the following outputs:

Sentiment: Positive
Confidence Score: 1.0 (indicating very high certainty)
Named Entities: Amazon (identified as an organization)
Summary: The generated summary captures the overall positive opinion of the user about the product, highlighting ease of use and satisfaction.

These results demonstrate that the model is capable of understanding the sentiment, extracting key entities, and summarizing the text effectively. The high confidence score indicates that the sentiment model is highly certain about its prediction. Additionally, the NER model successfully identifies relevant entities from the text, and the summarization model provides a concise representation of the original review.

# Conclusion

This project successfully demonstrates the application of modern Natural Language Processing techniques using pre-trained transformer models. By integrating sentiment analysis, named entity recognition, and text summarization into a single pipeline, a comprehensive AI text analysis tool was developed.

The results show that transformer-based models such as BERT and T5 can effectively understand the context of text and generate meaningful outputs without requiring extensive preprocessing or training. Compared to traditional NLP methods, these models provide better contextual understanding and more accurate results.

However, it is also observed that abstractive summarization may sometimes produce repetitive or simplified sentences, especially when working with short input texts or lightweight models. Despite this limitation, the overall performance of the system remains strong and suitable for practical applications.

In conclusion, this project highlights the power of transfer learning and transformer models in solving real-world NLP problems. The developed tool can be extended further for applications such as customer feedback analysis, business insights, and automated text processing systems.